# Medical Image Analysis for Disease Detection

This notebook demonstrates how to use the medical image analysis system for skin cancer detection using the HAM10000 dataset.

## Features
- Multiple deep learning architectures (ResNet, DenseNet, Vision Transformers)
- Comprehensive data preprocessing and augmentation
- Advanced training pipeline with early stopping
- Detailed evaluation metrics and visualizations
- Medical report generation

In [ ]:
# Import required libraries
import os
import sys
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Add src to path
sys.path.append('../src')

from src.models.architectures import create_model, MODEL_CONFIGS
from src.data.data_loader import create_data_loaders, visualize_samples
from src.training.trainer import MedicalImageTrainer
from src.utils.visualization import (
    set_random_seeds, get_device, count_parameters,
    create_classification_report_plot, plot_roc_curves,
    create_interactive_plots, create_medical_report
)

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Setup and Configuration

In [ ]:
# Set random seeds for reproducibility
set_random_seeds(42)

# Get device
device = get_device()

# Load configuration
config_path = '../configs/skin_cancer_config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"Dataset: {config['data']['dataset_type']}")
print(f"Model: {config['model']['name']}")
print(f"Classes: {len(config['class_names'])}")
print(f"Device: {device}")

## 2. Data Loading and Exploration

**Note**: To run this notebook, you need to download the Skin Cancer MNIST dataset from:
https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000

Update the `data_dir` in the configuration file to point to your dataset location.

In [ ]:
# Update data directory (modify this path)
# config['data']['data_dir'] = '/path/to/your/skin_cancer_dataset'

# For demonstration, we'll create a mock setup
print("To use this notebook with real data:")
print("1. Download the HAM10000 dataset from Kaggle")
print("2. Update the data_dir in the config file")
print("3. Ensure the dataset structure matches the expected format")

# Mock data info for demonstration
class_descriptions = {
    'mel': 'Melanoma - Most dangerous type of skin cancer',
    'nv': 'Melanocytic nevus - Common mole, usually benign',
    'bcc': 'Basal cell carcinoma - Most common skin cancer',
    'akiec': 'Actinic keratosis - Precancerous lesion',
    'bkl': 'Benign keratosis - Non-cancerous growth',
    'df': 'Dermatofibroma - Benign skin nodule',
    'vasc': 'Vascular lesion - Blood vessel abnormality'
}

print("\nSkin Cancer Types:")
for code, desc in class_descriptions.items():
    print(f"  {code}: {desc}")

In [ ]:
# Uncomment and run this cell when you have the dataset
"""
try:
    # Create data loaders
    train_loader, val_loader, test_loader = create_data_loaders(
        dataset_type=config['data']['dataset_type'],
        data_dir=config['data']['data_dir'],
        batch_size=config['training']['batch_size'],
        image_size=tuple(config['data']['image_size'])
    )
    
    print(f"Training samples: {len(train_loader.dataset)}")
    print(f"Validation samples: {len(val_loader.dataset)}")
    print(f"Test samples: {len(test_loader.dataset)}")
    
    # Visualize sample images
    print("\nSample images from the dataset:")
    visualize_samples(train_loader, config['class_names'], num_samples=8)
    
except FileNotFoundError:
    print("Dataset not found. Please download and set up the dataset first.")
"""

## 3. Model Architecture Exploration

In [ ]:
# Explore different model architectures
model_comparisons = {}

for config_type, models in MODEL_CONFIGS.items():
    print(f"\n{config_type.upper()} Configuration:")
    print("-" * 40)
    
    for model_type, model_name in models.items():
        try:
            model = create_model(
                model_name=model_name,
                num_classes=config['model']['num_classes'],
                pretrained=True
            )
            
            model_info = count_parameters(model)
            
            print(f"  {model_type.capitalize()}: {model_name}")
            print(f"    Parameters: {model_info['total_parameters']:,}")
            print(f"    Size: {model_info['total_parameters'] * 4 / (1024**2):.1f} MB")
            
            model_comparisons[f"{config_type}_{model_type}"] = model_info
            
        except Exception as e:
            print(f"  {model_type.capitalize()}: Error - {e}")

In [ ]:
# Visualize model complexity comparison
if model_comparisons:
    model_names = list(model_comparisons.keys())
    param_counts = [model_comparisons[name]['total_parameters'] for name in model_names]
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(range(len(model_names)), param_counts, alpha=0.7)
    plt.xlabel('Model Architecture')
    plt.ylabel('Number of Parameters')
    plt.title('Model Complexity Comparison')
    plt.xticks(range(len(model_names)), model_names, rotation=45, ha='right')
    
    # Add value labels on bars
    for bar, count in zip(bars, param_counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(param_counts)*0.01,
                f'{count/1e6:.1f}M', ha='center', va='bottom')
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Training Pipeline Demonstration

This section shows how to set up and run the training pipeline. For a full training run, use the command-line script `train.py`.

In [ ]:
# Create a model for demonstration
model = create_model(
    model_name=config['model']['name'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
    dropout_rate=config['model']['dropout_rate']
)

# Get model info
model_info = count_parameters(model)
print(f"Model: {config['model']['name']}")
print(f"Total parameters: {model_info['total_parameters']:,}")
print(f"Trainable parameters: {model_info['trainable_parameters']:,}")
print(f"Model size: {model_info['total_parameters'] * 4 / (1024**2):.1f} MB")

# Create trainer
trainer = MedicalImageTrainer(
    model=model,
    device=device,
    num_classes=config['model']['num_classes'],
    class_names=config['class_names']
)

print("\nTraining pipeline ready!")
print("To train the model, run: python train.py --config configs/skin_cancer_config.yaml")

## 5. Demo Training Results Visualization

This section demonstrates what the training results would look like with mock data.

In [ ]:
# Create mock training history for demonstration
np.random.seed(42)
epochs = 20

# Simulate realistic training curves
train_loss = 2.0 * np.exp(-np.arange(epochs) * 0.15) + 0.1 + np.random.normal(0, 0.05, epochs)
val_loss = 2.2 * np.exp(-np.arange(epochs) * 0.12) + 0.15 + np.random.normal(0, 0.08, epochs)
train_acc = 100 * (1 - np.exp(-np.arange(epochs) * 0.2)) + np.random.normal(0, 2, epochs)
val_acc = 100 * (1 - np.exp(-np.arange(epochs) * 0.18)) + np.random.normal(0, 3, epochs)

# Ensure values are realistic
train_loss = np.clip(train_loss, 0.05, 3.0)
val_loss = np.clip(val_loss, 0.1, 3.5)
train_acc = np.clip(train_acc, 10, 95)
val_acc = np.clip(val_acc, 5, 90)

# Plot training history
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss plot
axes[0, 0].plot(train_loss, label='Training Loss', linewidth=2)
axes[0, 0].plot(val_loss, label='Validation Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy plot
axes[0, 1].plot(train_acc, label='Training Accuracy', linewidth=2)
axes[0, 1].plot(val_acc, label='Validation Accuracy', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Learning rate plot
lr_schedule = 0.001 * (0.5 ** (np.arange(epochs) // 7))
axes[1, 0].plot(lr_schedule, linewidth=2, color='orange')
axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Learning Rate')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Overfitting indicator
loss_diff = val_loss - train_loss
axes[1, 1].plot(loss_diff, linewidth=2, color='red')
axes[1, 1].set_title('Overfitting Indicator', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Val Loss - Train Loss')
axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final Training Accuracy: {train_acc[-1]:.2f}%")
print(f"Final Validation Accuracy: {val_acc[-1]:.2f}%")

## 6. Demo Evaluation Results

In [ ]:
# Create mock evaluation results
np.random.seed(42)
n_classes = len(config['class_names'])
n_samples = 1000

# Generate mock predictions and true labels
true_labels = np.random.choice(n_classes, n_samples)
# Make predictions somewhat accurate
predictions = true_labels.copy()
# Add some errors
error_indices = np.random.choice(n_samples, int(n_samples * 0.2), replace=False)
predictions[error_indices] = np.random.choice(n_classes, len(error_indices))

# Create confusion matrix
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(true_labels, predictions)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=config['class_names'],
           yticklabels=config['class_names'],
           cbar_kws={'label': 'Number of Samples'})
plt.title('Confusion Matrix - Skin Cancer Classification', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Print classification report
print("Classification Report:")
print("=" * 50)
print(classification_report(true_labels, predictions, target_names=config['class_names']))

In [ ]:
# Create per-class performance visualization
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average=None)

# Create performance chart
x = np.arange(len(config['class_names']))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 8))
bars1 = ax.bar(x - width, precision, width, label='Precision', alpha=0.8, color='skyblue')
bars2 = ax.bar(x, recall, width, label='Recall', alpha=0.8, color='lightgreen')
bars3 = ax.bar(x + width, f1, width, label='F1-Score', alpha=0.8, color='lightcoral')

ax.set_xlabel('Disease Classes', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Class Performance Metrics', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(config['class_names'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)

# Add value labels on bars
def add_value_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
               f'{height:.2f}', ha='center', va='bottom', fontsize=9)

add_value_labels(bars1)
add_value_labels(bars2)
add_value_labels(bars3)

plt.tight_layout()
plt.show()

## 7. Medical Report Generation

In [ ]:
# Generate a sample medical report
mock_results = {
    'accuracy': 0.847,
    'precision': 0.851,
    'recall': 0.847,
    'f1_score': 0.848,
    'roc_auc': 0.923,
    'per_class_metrics': {
        'precision': precision.tolist(),
        'recall': recall.tolist(),
        'f1_score': f1.tolist()
    }
}

report = create_medical_report(
    results=mock_results,
    model_info=model_info,
    config=config,
    class_names=config['class_names']
)

print(report)

## 8. Clinical Impact and Usage Guidelines

### Healthcare Impact

This AI system demonstrates significant potential for improving healthcare outcomes:

1. **Early Detection**: Automated screening can help identify suspicious lesions earlier
2. **Diagnostic Assistance**: Supports dermatologists in making more accurate diagnoses
3. **Accessibility**: Can be deployed in areas with limited specialist availability
4. **Consistency**: Reduces inter-observer variability in diagnosis
5. **Efficiency**: Enables rapid processing of large numbers of cases

### Usage Recommendations

1. **Screening Tool**: Use as a first-line screening tool, not for final diagnosis
2. **Professional Oversight**: Always require review by qualified dermatologists
3. **Patient Education**: Inform patients about AI assistance in their care
4. **Continuous Learning**: Regularly update models with new data
5. **Quality Assurance**: Implement robust validation and monitoring systems

### Ethical Considerations

- Ensure patient privacy and data security
- Address potential biases in training data
- Maintain transparency about AI involvement
- Provide clear explanations of AI limitations
- Ensure equitable access to AI-assisted care

## 9. Next Steps

To use this system with real data:

1. **Download Datasets**:
   - [Skin Cancer MNIST (HAM10000)](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000)
   - [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)

2. **Setup Data Directory**:
   ```
   data/
   ├── skin_cancer/
   │   ├── HAM10000_metadata.csv
   │   └── images/
   └── brain_tumor/
       ├── glioma/
       ├── meningioma/
       ├── notumor/
       └── pituitary/
   ```

3. **Train Models**:
   ```bash
   python train.py --config configs/skin_cancer_config.yaml
   python train.py --config configs/brain_tumor_config.yaml
   ```

4. **Run Inference**:
   ```bash
   python inference.py --model-path checkpoints/best_model.pth \
                      --config-path configs/skin_cancer_config.yaml \
                      --image-path path/to/image.jpg \
                      --visualize
   ```

5. **Experiment with Different Models**:
   - Try different architectures (ResNet, DenseNet, ViT)
   - Adjust hyperparameters
   - Implement ensemble methods
   - Add advanced augmentation techniques